In [ ]:
import sys
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import numpy as np
import pandas as pd
import shapely
import spatialdata as sd
import spatialdata_plot
import zarr
from anndata.io import read_elem
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from shapely import affinity
from spatialdata.models import Image2DModel, PointsModel, ShapesModel
from spatialdata.transformations import Affine, Scale, Sequence, get_transformation, set_transformation

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

warnings.filterwarnings("ignore", message="SpatialData is not stored in the most current format")
warnings.filterwarnings("ignore", message="The object has `points` element")

In [ ]:
XENIUM_ZARR = Path("..").resolve() / "xenium_data_mouse" / "integrated" / "integrated_mouse.zarr"
ANNOTATED_ZARR = Path("data") / "integrated_mouse_annotated.zarr"
OUT_DIR = Path("results") / "05_mmp_mapping"

FRAMES = {
    "γ-ray Sham (ROI 8)": ((4310, 16140), 5, 0),
    "C-ion 3.5 d (ROI 6)": ((1810, 15440), 30, 90),
    "γ-ray 3.5 d (ROI 13)": ((7800, 14400), 350, 0),
}
FRAME_SIZE = 700
SCALE_BAR = 50

GRID_SECTIONS = ["C-ion 3.5 d (ROI 6)", "γ-ray 3.5 d (ROI 13)"]
GRID_WIDTH = 180 * fs.MM

CHANNEL_COLORS = {"DAPI": "#3D7BFF", "ATP1A1/CD45/E-Cadherin": "#FF00FF", "alphaSMA/Vimentin": "#00FF00"}
PERCENTILE = (1, 99)
ZOOM_BRIGHTNESS = 0.5
DOWNSAMPLE = 8

CELLTYPE_KEY = "cell_subtype"
CELL_TYPES = ["Adamdec1+ fibroblast", "Macrophage"]
GENE_COLORS = {"Mmp12": "#FFE600", "Mmp13": "#FF2A2A"}
QV = 20

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

obs = read_elem(zarr.open_group(ANNOTATED_ZARR, mode="r")["obs"])[["section", "condition", "slide", CELLTYPE_KEY]]
sdata = sd.read_zarr(XENIUM_ZARR, selection=("images", "shapes", "points"))

def rotation(center, deg):
    (cx, cy), t = center, np.deg2rad(deg)
    c, s = np.cos(t), np.sin(t)
    return Affine([[c, -s, cx - c * cx + s * cy], [s, c, cy - s * cx - c * cy], [0, 0, 1]],
                  input_axes=("x", "y"), output_axes=("x", "y"))

def frame_polygon(center, size, angle):
    half = size / 2
    return affinity.rotate(shapely.box(center[0] - half, center[1] - half, center[0] + half, center[1] + half), -angle, origin=center)

transcripts = sdata["transcripts"]
sections = {}
for section, (center, angle, turn) in FRAMES.items():
    obs_sec = obs[obs["section"] == section]
    roi_id = f"{obs_sec['condition'].iloc[0]}_{obs_sec['slide'].iloc[0]}"
    image = sdata[f"morphology_focus_{roi_id}"]

    image_small = image.coarsen(y=DOWNSAMPLE, x=DOWNSAMPLE, boundary="trim").mean().compute()
    image_small = Image2DModel.parse(image_small.values, dims=("c", "y", "x"), c_coords=image.c.values, transformations={
        "global": Sequence([Scale([DOWNSAMPLE, DOWNSAMPLE], axes=("x", "y")), get_transformation(image, "global")])})

    obs_shown = obs_sec[obs_sec[CELLTYPE_KEY].isin(CELL_TYPES)]
    boundaries = ShapesModel.parse(
        gpd.GeoDataFrame(obs_shown[[CELLTYPE_KEY]], geometry=sdata["cell_boundaries"].geometry.loc[obs_shown.index].values),
        transformations={"global": get_transformation(sdata["cell_boundaries"], "global")})

    genes = transcripts[(transcripts["roi_id"] == roi_id) & transcripts["feature_name"].isin(list(GENE_COLORS))
                        & (transcripts["qv"] >= QV)][["x", "y", "feature_name"]].compute()
    genes["feature_name"] = pd.Categorical(genes["feature_name"].astype(str), categories=list(GENE_COLORS))
    points = PointsModel.parse(genes.reset_index(drop=True), coordinates={"x": "x", "y": "y"},
                               transformations={"global": get_transformation(transcripts, "global")})

    sec = sections[section] = sd.SpatialData(
        images={"image": image, "image_small": image_small},
        shapes={"boundaries": boundaries, "frame": ShapesModel.parse(gpd.GeoDataFrame(geometry=[frame_polygon(center, FRAME_SIZE, angle)]))},
        points={"transcripts": points})
    for name in ("image", "boundaries", "transcripts"):
        set_transformation(sec[name], Sequence([get_transformation(sec[name], "global"), rotation(center, angle - turn)]), "zoom")
    print(section, len(obs_sec), "cells,", len(obs_shown), "shown,", genes["feature_name"].value_counts().to_dict())

In [ ]:
limits = {}
for channel in CHANNEL_COLORS:
    pixels = np.concatenate([sec["image_small"].sel(c=channel).values.ravel() for sec in sections.values()])
    limits[channel] = np.percentile(pixels[pixels > 0], PERCENTILE)

def render_image(sec, name, alpha=1):
    return sec.pl.render_images(name, channel=list(CHANNEL_COLORS), palette=list(CHANNEL_COLORS.values()),
                                norm=[Normalize(*limits[channel]) for channel in CHANNEL_COLORS], scale="full", alpha=alpha)

def finish_image(ax):
    ax.images[0].set_interpolation("none")
    ax.set_facecolor("black")

def coordinate_axes(ax, step):
    ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(step))
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(step))
    ax.tick_params(axis="x", labelrotation=90)
    ax.grid(color="white", lw=0.25, alpha=0.5)

def draw_cells(sec, point_size):
    if "boundaries" in sec.shapes:
        sec = sec.pl.render_shapes("boundaries", color=CELLTYPE_KEY, palette=celltype_colors, fill_alpha=1,
                                   outline_alpha=1, outline_color="black", outline_width=0.1)
    if "transcripts" in sec.points:
        sec = sec.pl.render_points("transcripts", color="feature_name", palette=GENE_COLORS, size=point_size, method="matplotlib")
    return sec

def crop(sec, center, size):
    half = size / 2
    return sec.filter_by_coordinate_system("zoom").query.bounding_box(
        axes=["x", "y"], min_coordinate=[center[0] - half, center[1] - half],
        max_coordinate=[center[0] + half, center[1] + half], target_coordinate_system="zoom")

In [ ]:
for section, sec in sections.items():
    fig, ax = plt.subplots(figsize=(fs.COL1, fs.COL1), layout="constrained")
    (render_image(sec, "image_small")
     .pl.render_shapes("frame", fill_alpha=0, outline_alpha=1, outline_color=fs.YELLOW, outline_width=0.75)
     .pl.show(coordinate_systems="global", ax=ax, legend_loc=None, colorbar=False))
    finish_image(ax)

    extent = sd.get_extent(sec["image_small"])
    ax.set(xlim=extent["x"], ylim=extent["y"][::-1], aspect="equal", xlabel="x (µm)", ylabel="y (µm)", title=section)
    coordinate_axes(ax, 500)
    plt.show()

In [ ]:
INSETS = {
    "γ-ray Sham (ROI 8)": (4550, 16320),
    "C-ion 3.5 d (ROI 6)": (1750, 15730),
    "γ-ray 3.5 d (ROI 13)": (7980, 14630),
}
INSET_SIZE = 200
INSET_SCALE_BAR = 50

celltype_colors = dict(zip(CELL_TYPES, fs.palette(CELL_TYPES)))
legend_handles = ([Patch(fc=color, ec="black", lw=0.3, label=celltype) for celltype, color in celltype_colors.items()]
                  + [Line2D([], [], ls="", marker="o", ms=3, mew=0, color=color, label=fs.gene(gene)) for gene, color in GENE_COLORS.items()])

def add_legend(fig):
    fig.legend(handles=legend_handles, loc="outside lower center", ncol=len(legend_handles), handlelength=1, handleheight=0.8)

def crop_global(sec, polygon):
    x0, y0, x1, y1 = polygon.bounds
    return sec.query.bounding_box(axes=["x", "y"], min_coordinate=[x0, y0], max_coordinate=[x1, y1], target_coordinate_system="global")

for section, (center, angle, _) in FRAMES.items():
    frame = frame_polygon(center, FRAME_SIZE, angle)
    inset = frame_polygon(INSETS[section], INSET_SIZE, angle)
    fig, axes = plt.subplots(1, 2, figsize=(fs.COL2, fs.COL2 / 2 + 10 * fs.MM), layout="constrained")
    for ax, polygon, point_size, step, title in [(axes[0], frame, 0.5, 100, f"{section}: frame {FRAME_SIZE} µm"),
                                                 (axes[1], inset, 2, 25, f"{section}: inset {INSET_SIZE} µm")]:
        draw_cells(render_image(crop_global(sections[section], polygon), "image", alpha=ZOOM_BRIGHTNESS), point_size).pl.show(
            coordinate_systems="global", ax=ax, legend_loc=None, colorbar=False)
        finish_image(ax)
        ax.plot(*frame.exterior.xy, color=fs.YELLOW, lw=0.75)
        ax.plot(*inset.exterior.xy, color="white", lw=0.75)
        x0, y0, x1, y1 = polygon.bounds
        ax.set(xlim=(x0, x1), ylim=(y1, y0), aspect="equal", xlabel="x (µm)", ylabel="y (µm)", title=title)
        coordinate_axes(ax, step)
    add_legend(fig)
    plt.show()

In [ ]:
def scale_bar(ax, length, loc=(0.05, 0.05), color="black"):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    x = min(x0, x1) + loc[0] * abs(x1 - x0)
    y = y0 + loc[1] * (y1 - y0)
    ax.plot([x, x + length], [y, y], color=color, lw=1.0, solid_capstyle="butt")
    ax.annotate(f"{length:g} µm", (x + length / 2, y), xytext=(0, 2), textcoords="offset points",
                ha="center", va="bottom", fontsize=6, color=color)

def finish_panel(ax, center, size, bar_length, label):
    half = size / 2
    ax.set(xlim=(center[0] - half, center[0] + half), ylim=(center[1] + half, center[1] - half), aspect="equal",
           xticks=[], yticks=[], ylabel="", title="")
    ax.set_xlabel(label, labelpad=3)
    ax.spines[:].set_visible(False)
    ax.apply_aspect()
    scale_bar(ax, bar_length, loc=(0.05, 0.04), color="white")

def panel_row():
    fig, axes = plt.subplots(1, len(GRID_SECTIONS), figsize=(GRID_WIDTH, GRID_WIDTH / len(GRID_SECTIONS) + 15 * fs.MM), layout="constrained")
    add_legend(fig)
    return fig, axes

fig_zoom, axes_zoom = panel_row()
fig_inset, axes_inset = panel_row()
for ax_zoom, ax_inset, section in zip(axes_zoom, axes_inset, GRID_SECTIONS):
    center, angle, turn = FRAMES[section]
    matrix = rotation(center, angle - turn).to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))
    inset_center = (matrix @ [*INSETS[section], 1])[:2]

    draw_cells(render_image(crop(sections[section], center, FRAME_SIZE), "image", alpha=ZOOM_BRIGHTNESS), 0.5).pl.show(
        coordinate_systems="zoom", ax=ax_zoom, legend_loc=None, colorbar=False)
    finish_image(ax_zoom)
    finish_panel(ax_zoom, center, FRAME_SIZE, SCALE_BAR, section)
    ax_zoom.add_patch(Rectangle(inset_center - INSET_SIZE / 2, INSET_SIZE, INSET_SIZE, fill=False, ec="white", lw=0.6))

    draw_cells(render_image(crop(sections[section], inset_center, INSET_SIZE), "image", alpha=ZOOM_BRIGHTNESS), 2).pl.show(
        coordinate_systems="zoom", ax=ax_inset, legend_loc=None, colorbar=False)
    finish_image(ax_inset)
    finish_panel(ax_inset, inset_center, INSET_SIZE, INSET_SCALE_BAR, f"{section}, inset")

fs.save(fig_zoom, OUT_DIR / f"zoom_{FRAME_SIZE}um")
fs.save(fig_inset, OUT_DIR / f"inset_{INSET_SIZE}um")
plt.show()